In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

In [2]:
# Load cleaned data
df = pd.read_csv(data_root / 'CBOS_data.csv')

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_35327/246268884.py:2: DtypeWarning: Columns (14,15,20,26,27,28,32,33,34,39,40,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_root / 'CBOS_data.csv')


In [3]:
# Get rid of invalid sex
df['sex'] = df['sex'].apply(lambda x: np.nan if x not in [1.0, 2.0] else x)

In [4]:
# Handle age
df['age'] = np.nan
for id, row in df.iterrows():
    age = row['survey_year'] - row['year_born']
    if age < 0:
        print(f"Warning: Negative age for id {id} (survey year: {row['survey_year']}, survey month: {row['survey_month']}, year born: {row['year_born']})")
    elif age > 200:
        yb = row['year_born']
        if len(str(yb)) == 3:
            yb = '0' + str(yb)
        else:
            yb = str(yb)    
        
        full_year = str('19' + yb)
        
        if len(full_year) == 6:
            full_year = full_year[:-2]
        
        intyear = int(full_year)
        df.at[id, 'age'] = row['survey_year'] - intyear
    else:
        df.at[id, 'age'] = age

In [5]:
# Unique pairs sex and sex_L
display(df[['sex', 'sex_L']].drop_duplicates())
SEX_MAPPING = {1.0: 'Mężczyzna', 2.0: 'Kobieta'}
df['sex_L'] = df['sex'].map(SEX_MAPPING)
display(df[['sex', 'sex_L']].drop_duplicates())

,sex,sex_L
0,2.0,Kobieta
3,1.0,Mężczyzna
1184,NaN,NaN
10318,1.0,Mężczyźni
10320,2.0,Kobiety
30689,NaN,Brak danych / Odmowa odpowiedzi
32867,NaN,BRAK DANYCH / Odmowa odpowiedzi
36311,NaN,Brak danych/ Odmowa odpowiedzi
219430,2.0,Kobieta
219431,1.0,Mężczyzna


,sex,sex_L
0,2.0,Kobieta
3,1.0,Mężczyzna
1184,NaN,NaN


In [6]:
# Location variable
df['location_old'] = df['location_old'].apply(lambda x: np.nan if x not in [float(x) for x in range(1,50)] else x)
for idx, row in df.iterrows():
    if np.isnan(row['location_old']):
        df.at[idx, 'location_old_L'] = np.nan
    if row['location_old'] == 2.0:
        df.at[idx, 'location_old_L'] = 'bialskopodlaskie'
df['location_old_L'] = df['location_old_L'].str.lower()

display(df[['location_old', 'location_old_L']].drop_duplicates())

,location_old,location_old_L
0,1.0,warszawskie
2,5.0,bydgoskie
4,NaN,NaN
7,3.0,białostockie
8,2.0,bialskopodlaskie
11,4.0,bielskie
1486,29.0,pilskie
1504,31.0,płockie
1525,38.0,skierniewickie
1558,40.0,suwalskie


In [7]:
# Location variable
df['location_new'] = df['location_new'].apply(lambda x: np.nan if x not in [float(x) for x in range(1,50)] else x)
for idx, row in df.iterrows():
    if np.isnan(row['location_new']):
        df.at[idx, 'location_new_L'] = np.nan
df['location_new_L'] = df['location_new_L'].str.lower()
df['location_new_L'] = df['location_new_L'].str.strip()

display(df[['location_new', 'location_new_L']].drop_duplicates())

,location_new,location_new_L
0,NaN,NaN
122965,6.0,małopolskie
123005,12.0,śląskie
123032,11.0,pomorskie
123046,16.0,zachodniopomorskie
123139,7.0,mazowieckie
123167,5.0,łódzkie
123244,3.0,lubelskie
123373,2.0,kujawsko-pomorskie
123394,15.0,wielkopolskie


In [ ]:
# City size
y = pd.DataFrame()
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    df_sub['city_size_L'] = df_sub['city_size_L'].str.strip()
    df_sub['city_size_L'] = df_sub['city_size_L'].str.lower()
    x = df_sub[['city_size', 'city_size_L']].drop_duplicates().sort_values(by='city_size').reset_index(drop=True)
    
    if x.equals(y):
        print(f"File {file} has consistent city_size and city_size_L values.")
    else:
        x = pd.DataFrame(x)
        #display(x)
    y = x
    

,city_size,city_size_L
0,1.0,wieś
1,2.0,miasto do 5 tys. mieszkańców
2,3.0,miasto 6 – 10 tys. mieszkańców
3,4.0,miasto 11 – 20 tys. mieszkańców
4,5.0,miasto 21 – 50 tys. mieszkańców
5,6.0,miasto 51 – 100 tys. mieszkańców
6,7.0,miasto 101 – 200 tys. mieszkańców
7,8.0,miasto 201 – 500 tys. mieszkańców
8,9.0,miasto powyżej 500 tys. mieszkańców
9,NaN,NaN


File CBOS_2_02_1990.sav has consistent city_size and city_size_L values.


,city_size,city_size_L
0,1.0,wieś
1,2.0,miasto do 5 tys. mieszkańców
2,3.0,miasto 6 – 10 tys. mieszkańców
3,4.0,miasto 11 – 20 tys. mieszkańców
4,5.0,miasto 21 – 40 tys. mieszkańców
5,6.0,miasto 41 – 100 tys. mieszkańców
6,7.0,miasto 101 – 200 tys. mieszkańców
7,8.0,miasto 201 – 500 tys. mieszkańców
8,9.0,miasto powyżej 500 tys. mieszkańców
9,NaN,NaN


File CBOS_4_04_1990.sav has consistent city_size and city_size_L values.
File CBOS_5_05_1990.sav has consistent city_size and city_size_L values.
File CBOS_6_06_1990.sav has consistent city_size and city_size_L values.


,city_size,city_size_L
0,1.0,wsią
1,2.0,miastem: poniżej 20 tys. mieszkańców
2,3.0,miastem: od 20 do 100 tys. mieszkańców
3,4.0,miastem: od 100 do 500 tys. mieszkańców
4,5.0,miastem: 500 tys. mieszkańców lub więcej
5,NaN,NaN


...

,city_size,city_size_L
0,1.0,wieś
1,2.0,miasto do 19 999
2,3.0,miasto od 20 000 do 49 999
3,4.0,miasto od 50 000 do 99 999
4,5.0,miasto od 100 000 do 499 999
5,6.0,miasto 500 000 i więcej


File CBOS_270_11_2012.sav has consistent city_size and city_size_L values.
File CBOS_271_12_2012.sav has consistent city_size and city_size_L values.
File CBOS_272_01_2013.sav has consistent city_size and city_size_L values.
File CBOS_273_02_2013.sav has consistent city_size and city_size_L values.
File CBOS_274_03_2013.sav has consistent city_size and city_size_L values.
File CBOS_275_04_2013.sav has consistent city_size and city_size_L values.
File CBOS_276_05_2013.sav has consistent city_size and city_size_L values.
File CBOS_277_06_2013.sav has consistent city_size and city_size_L values.
File CBOS_278_07_2013.sav has consistent city_size and city_size_L values.
File CBOS_279_08_2013.sav has consistent city_size and city_size_L values.
File CBOS_280_09_2013.sav has consistent city_size and city_size_L values.
File CBOS_281_10_2013.sav has consistent city_size and city_size_L values.
File CBOS_282_11_2013.sav has consistent city_size and city_size_L values.
File CBOS_283_12_2013.sav

,city_size,city_size_L
0,1.0,wieś
1,2.0,miasto do 19 999
2,4.0,miasto od 50 000 do 99 999
3,5.0,miasto od 100 000 do 499 999
4,6.0,miasto 500 000 i więcej


,city_size,city_size_L
0,1.0,wieś
1,2.0,miasto do 19 999
2,3.0,miasto od 20 000 do 49 999
3,4.0,miasto od 50 000 do 99 999
4,5.0,miasto od 100 000 do 499 999
5,6.0,miasto 500 000 i więcej


File CBOS_299_04_2015.sav has consistent city_size and city_size_L values.
File CBOS_300_05_2015.sav has consistent city_size and city_size_L values.
File CBOS_301_06_2015.sav has consistent city_size and city_size_L values.
File CBOS_302_07_2015.sav has consistent city_size and city_size_L values.
File CBOS_303_08_2015.sav has consistent city_size and city_size_L values.
File CBOS_304_09_2015.sav has consistent city_size and city_size_L values.
File CBOS_305_10_2015.sav has consistent city_size and city_size_L values.
File CBOS_306_11_2015.sav has consistent city_size and city_size_L values.
File CBOS_307_12_2015.sav has consistent city_size and city_size_L values.
File CBOS_308_01_2016.sav has consistent city_size and city_size_L values.
File CBOS_309_02_2016.sav has consistent city_size and city_size_L values.
File CBOS_310_03_2016.sav has consistent city_size and city_size_L values.
File CBOS_311_04_2016.sav has consistent city_size and city_size_L values.
File CBOS_312_05_2016.sav

In [9]:
CITY_SIZE_MAPPING = {
    1.0: 'wieś',
    2.0: 'do 20 tys. mieszkańców',
    3.0: '20-100 tys. mieszkańców',
    4.0: '100-500 tys. mieszkańców',
    5.0: 'powyżej 500 tys. mieszkańców',
    np.nan: ''
}
KLM6_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0],
    3.0: [3.0, 4.0],
    4.0: [5.0],
    5.0: [6.0],
    np.nan: [8.0]
}
KLM7_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0],
    3.0: [3.0, 4.0],
    4.0: [5.0, 6.0],
    5.0: [7.0]
}
KLM9_NEW_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0, 3.0, 4.0, 5.0],
    3.0: [6.0, 7.0],
    4.0: [8.0],
    5.0: [9.0]
}
KLM9_OLD_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0, 3.0, 4.0],
    3.0: [5.0, 6.0],
    4.0: [7.0, 8.0],
    5.0: [9.0]
}

# CBOS_49_05_1994.sav has a different city size variable. It is not possible to map it to the same city size variable as in other files, so we will keep it as is.

In [10]:
def in_which_key(value, mapping):
    for key, values in mapping.items():
        if value in values:
            return key
    return np.nan

In [11]:
df['cs'] = np.nan
df['cs_L'] = ''

for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    values = df_sub['city_size'].dropna().unique()
    has_7 = 7.0 in values
    has_6 = 6.0 in values
    has_8 = 8.0 in values
    has_9 = 9.0 in values
    if has_6 and not has_7 and not has_8 and not has_9:
        mapping = KLM6_CITY_SIZE_MAPPING
    elif has_6 and has_7 and not has_8 and not has_9:
        mapping = KLM7_CITY_SIZE_MAPPING
    elif has_6 and not has_7 and has_8 and not has_9:
        mapping = KLM6_CITY_SIZE_MAPPING
    elif has_6 and has_7 and has_8 and has_9:
        if int(file.split('_')[1]) in list(range(1,7)):
            mapping = KLM9_OLD_CITY_SIZE_MAPPING
        else:
            mapping = KLM9_NEW_CITY_SIZE_MAPPING
    else:
        mapping = {1.0: [1.0], 2.0: [2.0], 3.0: [3.0], 4.0: [4.0], 5.0: [5.0], np.nan: [np.nan]}
    
    for idx, row in df_sub.iterrows():
        if np.isnan(row['city_size']):
            df.at[idx, 'cs'] = np.nan
            df.at[idx, 'cs_L'] = np.nan
        else:
            df.at[idx, 'cs'] = in_which_key(row['city_size'], mapping)

df['cs_L'] = df['cs'].map(CITY_SIZE_MAPPING)

In [12]:
# City size
y = pd.DataFrame()
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    df_sub['education_L'] = df_sub['education_L'].str.strip()
    df_sub['education_L'] = df_sub['education_L'].str.lower()
    x = df_sub[['education', 'education_L']].drop_duplicates().sort_values(by='education').reset_index(drop=True)
    for idx, row in x.iterrows():
        # Drop the rows that have np.nan in both education and education_L
        if pd.isna(row['education']) and pd.isna(row['education_L']):
            x = x.drop(idx)
    
    if x.equals(y):
        print(f"File {file} has consistent education and education_L values.")
    else:
        x = pd.DataFrame(x)
        #display(x)
    y = x
    

File CBOS_2_02_1990.sav has consistent education and education_L values.
File CBOS_3_03_1990.sav has consistent education and education_L values.
File CBOS_4_04_1990.sav has consistent education and education_L values.
File CBOS_5_05_1990.sav has consistent education and education_L values.
File CBOS_6_06_1990.sav has consistent education and education_L values.
File CBOS_7_07_1990.sav has consistent education and education_L values.
File CBOS_9_10_1990.sav has consistent education and education_L values.
File CBOS_10_11_1990.sav has consistent education and education_L values.
File CBOS_11_12_1990.sav has consistent education and education_L values.
File CBOS_12_01_1991.sav has consistent education and education_L values.
File CBOS_13_02_1991.sav has consistent education and education_L values.
File CBOS_14_03_1991.sav has consistent education and education_L values.
File CBOS_15_04_1991.sav has consistent education and education_L values.
File CBOS_16_05_1991.sav has consistent educa

In [13]:
EDUCATION_1990_MAP = {
    1.0: 'podstawowe nieukończone i bez wykształcenia',
    2.0: 'podstawowe',
    3.0: 'zasadnicze zawodowe',
    4.0: 'średnie',
    5.0: 'wyższe',
    np.nan: ''
}
# Till CBOS_117_02_2000.sav and from CBOS_119_04_2000.sav till CBOS_207_08_2007.sav
EDUCATION_1990_9_MAP = {
    1.0: [1.0],
    2.0: [2.0],
    3.0: [3.0],
    4.0: [4.0, 5.0, 6.0, 7.0, 8.0],
    5.0: [9.0],
    np.nan: [98.0]
}
# In CBOS_118_03_2000.sav
EDUCATION_1990_11_1_MAP = {
    1.0: [1.0, 2.0],
    2.0: [3.0, 4.0],
    3.0: [5.0],
    4.0: [6.0, 7.0, 8.0, 9.0, 10.0],
    5.0: [11.0],
    np.nan: [98.0]
}
# From CBOS_208_09_2007.sav till CBOS_220_09_2008.sav
EDUCATION_1990_11_2_MAP = {
    1.0: [1.0],
    2.0: [2.0, 3.0],
    3.0: [4.0],
    4.0: [5.0, 6.0, 7.0, 8.0, 9.0],
    5.0: [10.0, 11.0]
}
# From CBOS_221_10_2008.sav onwards
EDUCATION_1990_12_MAP = {
    1.0: [1.0],
    2.0: [2.0, 3.0],
    3.0: [4.0],
    4.0: [5.0, 6.0, 7.0, 8.0, 9.0],
    5.0: [10.0, 11.0, 12.0]
}

EDUCATION_2000_MAP = {
    1.0: 'gimnazjalne, podstawowe i niższe',
    2.0: 'zasadnicze zawodowe/branżowe',
    3.0: 'średnie ogólnokształcące',
    4.0: 'policealne oraz średnie zawodowe/branżowe',
    5.0: 'wyższe',
    np.nan: ''
}
# Till CBOS_117_02_2000.sav and from CBOS_119_04_2000.sav till CBOS_207_08_2007.sav
EDUCATION_2000_9_MAP = {
    1.0: [1.0, 2.0],
    2.0: [3.0],
    3.0: [4.0, 5.0, 8.0],
    4.0: [6.0, 7.0],
    5.0: [9.0],
    np.nan: [98.0]
}
# In CBOS_118_03_2000.sav
EDUCATION_2000_11_1_MAP = {
    1.0: [1.0, 2.0, 3.0, 4.0],
    2.0: [5.0],
    3.0: [6.0, 7.0, 10.0],
    4.0: [8.0, 9.0],
    5.0: [11.0],
    np.nan: [98.0]
}
# From CBOS_208_09_2007.sav till CBOS_220_09_2008.sav
EDUCATION_2000_11_2_MAP = {
    1.0: [1.0, 2.0, 3.0],
    2.0: [4.0],
    3.0: [5.0, 6.0, 9.0],
    4.0: [7.0, 8.0],
    5.0: [10.0, 11.0]
}
# From CBOS_221_10_2008.sav onwards
EDUCATION_2000_12_MAP = {
    1.0: [1.0, 2.0, 3.0],
    2.0: [4.0],
    3.0: [5.0, 6.0],
    4.0: [7.0, 8.0, 9.0],
    5.0: [10.0, 11.0, 12.0]
}


In [14]:
df['educ_1990'] = np.nan
df['educ_2000'] = np.nan
df['educ_1990_L'] = ''
df['educ_2000_L'] = ''

for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    if int(file.split('_')[1]) in list(range(1, 118)) + list(range(119, 208)):
        for idx, row in df_sub.iterrows():
            if pd.isna(row['education']) and pd.isna(row['education_L']):
                df_sub.at[idx, 'education'] = np.nan
            df.at[idx, 'educ_1990'] = in_which_key(row['education'], EDUCATION_1990_9_MAP)
            df.at[idx, 'educ_2000'] = in_which_key(row['education'], EDUCATION_2000_9_MAP)
    elif file.split('_')[1] == '118':
        for idx, row in df_sub.iterrows():
            if pd.isna(row['education']) and pd.isna(row['education_L']):
                df_sub.at[idx, 'education'] = np.nan
            df.at[idx, 'educ_1990'] = in_which_key(row['education'], EDUCATION_1990_11_1_MAP)
            df.at[idx, 'educ_2000'] = in_which_key(row['education'], EDUCATION_2000_11_1_MAP)
    elif int(file.split('_')[1]) in list(range(208, 221)):
        for idx, row in df_sub.iterrows():
            if pd.isna(row['education']) and pd.isna(row['education_L']):
                df_sub.at[idx, 'education'] = np.nan
            df.at[idx, 'educ_1990'] = in_which_key(row['education'], EDUCATION_1990_11_2_MAP)
            df.at[idx, 'educ_2000'] = in_which_key(row['education'], EDUCATION_2000_11_2_MAP)
    elif int(file.split('_')[1]) >= 221:
        for idx, row in df_sub.iterrows():
            if pd.isna(row['education']) and pd.isna(row['education_L']):
                df_sub.at[idx, 'education'] = np.nan
            df.at[idx, 'educ_1990'] = in_which_key(row['education'], EDUCATION_1990_12_MAP)
            df.at[idx, 'educ_2000'] = in_which_key(row['education'], EDUCATION_2000_12_MAP)
    else:
        print(f"File {file} has an unexpected survey number.")
    
df['educ_1990_L'] = df['educ_1990'].map(EDUCATION_1990_MAP)
df['educ_2000_L'] = df['educ_2000'].map(EDUCATION_2000_MAP)

In [15]:
# City size
y = pd.DataFrame()
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    df_sub['income_hh_OLD_T_L'] = df_sub['income_hh_OLD_T_L'].str.strip()
    df_sub['income_hh_OLD_T_L'] = df_sub['income_hh_OLD_T_L'].str.lower()
    df_sub['income_hh_NEW_T_L'] = df_sub['income_hh_NEW_T_L'].str.strip()
    df_sub['income_hh_NEW_T_L'] = df_sub['income_hh_NEW_T_L'].str.lower()
    df_sub['income_p_OLD_T_L'] = df_sub['income_p_OLD_T_L'].str.strip()
    df_sub['income_p_OLD_T_L'] = df_sub['income_p_OLD_T_L'].str.lower()
    df_sub['income_p_NEW_T_L'] = df_sub['income_p_NEW_T_L'].str.strip()
    df_sub['income_p_NEW_T_L'] = df_sub['income_p_NEW_T_L'].str.lower()
    
    col = 'income_hh_NEW_T'
    
    x = df_sub[[col, str(col+'_L')]].drop_duplicates().sort_values(by=col).reset_index(drop=True)
    for idx, row in x.iterrows():
        # Drop the rows that have np.nan in both education and education_L
        if pd.isna(row[col]) and pd.isna(row[str(col+'_L')]):
            x = x.drop(idx)
    
    if x.equals(y):
        print(f"File {file} has consistent {col} and {str(col + '_L')} values.")
    else:
        x = pd.DataFrame(x)
        #display(x)
    y = x
    

File CBOS_2_02_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_3_03_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_4_04_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_5_05_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_6_06_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_7_07_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_8_09_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_9_10_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_10_11_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_11_12_1990.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_12_01_1991.sav has consistent income_hh_NEW_T and income_hh_NEW_T_L values.
File CBOS_13_02_1991.sav has consistent income_hh_NEW_T and in

In [16]:
map = df[(df['survey_year'] == 2017) & (df['survey_month'] == 3)][['income_hh_NEW_T', 'income_hh_NEW_T_L']].drop_duplicates().sort_values(by='income_hh_NEW_T').reset_index(drop=True)

for idx, row in df.iterrows():
    if row['survey_year'] == 2017 and row['survey_month'] == 2:
        if pd.isna(row['income_hh_NEW_T']) and pd.isna(row['income_hh_NEW_T_L']):
            df.at[idx, 'income_hh_NEW_T'] = np.nan
            df.at[idx, 'income_hh_NEW_T_L'] = np.nan
        else:
            df.at[idx, 'income_hh_NEW_T_L'] = map[map['income_hh_NEW_T'] == row['income_hh_NEW_T']]['income_hh_NEW_T_L'].values[0]

In [17]:
# =============================================================================
# Impute income from thresholded (bracket) columns
#   income_hh_OLD_T  →  income_hh_OLD_IM
#   income_hh_NEW_T  →  income_hh_NEW_IM
#   income_p_OLD_T   →  income_p_OLD_IM
#   income_p_NEW_T   →  income_p_NEW_IM
#
# Rules:
#   (a, b)  → midpoint = (a + b) / 2
#   (0, a]  → gamma-shifted = a * 2/3
#   [a, ∞)  → Pareto tail (α≈3) = a * 3/2
#   technical / non-response / missing → NaN
# =============================================================================

import importlib
import income_imputer as iimp
importlib.reload(iimp)

df, manual_checks = iimp.impute_all_income_columns(df)

# Report manual-check records (category code present but label empty)
for col, indices in manual_checks.items():
    if indices:
        print(f"\n⚠ {col}: {len(indices)} rows with code but empty label (imputed as NaN):")
        sample = indices[:10]
        print(f"  Sample indices: {sample}")
        print(df.loc[sample, [col, col + '_L']].to_string())
    else:
        print(f"✓ {col}: all labels parsed successfully")

  income_hh_OLD_T → income_hh_OLD_IM: 7322 imputed, 351203 NaN, 0 need manual check
  income_hh_NEW_T → income_hh_NEW_IM: 26569 imputed, 331956 NaN, 0 need manual check
  income_p_OLD_T → income_p_OLD_IM: 14364 imputed, 344161 NaN, 0 need manual check
  income_p_NEW_T → income_p_NEW_IM: 16766 imputed, 341759 NaN, 0 need manual check
✓ income_hh_OLD_T: all labels parsed successfully
✓ income_hh_NEW_T: all labels parsed successfully
✓ income_p_OLD_T: all labels parsed successfully
✓ income_p_NEW_T: all labels parsed successfully


In [18]:
i=0
for idx, row in df.iterrows():
    if (np.isnan(row['income_hh']) and not np.isnan(row['income_hh_NEW'])):
        df.at[idx, 'income_hh'] = row['income_hh_NEW']
        i+=1
    elif (np.isnan(row['income_hh']) and not np.isnan(row['income_hh_NEW_IM'])):
        df.at[idx, 'income_hh'] = row['income_hh_NEW_IM']
        i+=1
    if (np.isnan(row['income_hh']) and not np.isnan(row['income_hh_OLD'])):
        df.at[idx, 'income_hh'] = row['income_hh_OLD'] / 10
        i+=1
    elif (np.isnan(row['income_hh']) and not np.isnan(row['income_hh_OLD_IM'])):
        df.at[idx, 'income_hh'] = row['income_hh_OLD_IM'] / 10000
        i+=1
    if (np.isnan(row['income_p']) and not np.isnan(row['income_p_NEW'])):
        df.at[idx, 'income_p'] = row['income_p_NEW']
        i+=1
    elif (np.isnan(row['income_p']) and not np.isnan(row['income_p_NEW_IM'])):
        df.at[idx, 'income_p'] = row['income_p_NEW_IM']
        i+=1
    if (np.isnan(row['income_p']) and not np.isnan(row['income_p_OLD'])):
        df.at[idx, 'income_p'] = row['income_p_OLD'] / 10
        i+=1
    elif (np.isnan(row['income_p']) and not np.isnan(row['income_p_OLD_IM'])):
        df.at[idx, 'income_p'] = row['income_p_OLD_IM'] / 10000
        i+=1
print(i)

65010


In [19]:
i=0
for idx, row in df.iterrows():
    if row['income_hh'] in [99999.0, 99997.0, 99998.0, 9999.0, 9999.4, 9999.5]:
        df.at[idx, 'income_hh'] = np.nan
        i += 1
    if row['income_hh'] in [99992.0]:
        df.at[idx, 'income_hh'] = 0.0
    if row['income_p'] in [99999.0, 99997.0, 99998.0, 9999.0, 9999.4, 9999.5]:
        df.at[idx, 'income_p'] = np.nan
        i += 1
    if row['income_p'] in [99992.0]:
        df.at[idx, 'income_p'] = 0.0

In [20]:
# Sol 
for idx, row in df.iterrows():
    label = row['sol_L']
    if str(label).strip().lower() in ['odmowa odpowiedzi', 'brak danych / odmowa odpowiedzi', 'trudno powiedzieć', 'brak danych / odmowa odpowiedzi', np.nan]:
        if row['sol'] in [7.0, 8.0, 9.0]:
            df.at[idx, 'sol'] = np.nan
            df.at[idx, 'sol_L'] = np.nan

In [21]:
# Sol
y = pd.DataFrame()
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    df_sub['sol_L'] = df_sub['sol_L'].str.strip()
    df_sub['sol_L'] = df_sub['sol_L'].str.lower()
    
    x = df_sub[['sol', 'sol_L']].drop_duplicates().sort_values(by='sol').reset_index(drop=True)
    for idx, row in x.iterrows():
        # Drop the rows that have np.nan in both education and education_L
        if pd.isna(row['sol']) and pd.isna(row['sol_L']):
            x = x.drop(idx)
    
    if x.equals(y):
        print(f"File {file} has consistent sol and sol_L values.")
    else:
        x = pd.DataFrame(x)
        #display(x)
    y = x
    
# odmowa odpowiedzi, brak danych / odmowa odpowiedzi, trudno powiedzieć

# CBOS_109_06_1999.sav and CBOS_105_02_1999.sav inverted

for idx, row in df.iterrows():
    if row['survey_file'] == 'CBOS_75_08_1996.sav':
        if row['sol'] ==7.0:
            df.at[idx, 'sol'] = np.nan
            df.at[idx, 'sol_L'] = np.nan
    if row['survey_file'] in ['CBOS_109_06_1999.sav', 'CBOS_105_02_1999.sav']:
        if row['sol'] == 1.0:
            df.at[idx, 'sol'] = 5.0
        elif row['sol'] == 2.0:
            df.at[idx, 'sol'] = 4.0

File CBOS_2_02_1990.sav has consistent sol and sol_L values.
File CBOS_3_03_1990.sav has consistent sol and sol_L values.
File CBOS_4_04_1990.sav has consistent sol and sol_L values.
File CBOS_5_05_1990.sav has consistent sol and sol_L values.
File CBOS_6_06_1990.sav has consistent sol and sol_L values.
File CBOS_9_10_1990.sav has consistent sol and sol_L values.
File CBOS_10_11_1990.sav has consistent sol and sol_L values.
File CBOS_11_12_1990.sav has consistent sol and sol_L values.
File CBOS_13_02_1991.sav has consistent sol and sol_L values.
File CBOS_17_06_1991.sav has consistent sol and sol_L values.
File CBOS_18_07_1991.sav has consistent sol and sol_L values.
File CBOS_21_10_1991.sav has consistent sol and sol_L values.
File CBOS_22_11_1991.sav has consistent sol and sol_L values.
File CBOS_27_09_1992.sav has consistent sol and sol_L values.
File CBOS_28_10_1992.sav has consistent sol and sol_L values.
File CBOS_29_11_1992.sav has consistent sol and sol_L values.
File CBOS_30_1

In [22]:
# Save dataframe 
df.to_csv(data_root / "CBOS_data_unified.csv", index = True)

In [ ]:
# Before:
print(f"Number of rows before dropping missing values: {len(df)}")
# mask when location_old is na and location_new is na
mask = df['location_old'].isna() & df['location_new'].isna()

# Drop rows with missing values in key variables
df = df[~df['sex'].isna()]
df = df[~df['age'].isna()]

# For location variables
df = df[~mask]

df = df[~df['city_size'].isna()]
df = df[~df['educ_1990'].isna()]
df = df[~df['educ_2000'].isna()]
df = df[~df['cs'].isna()]

# After:
print(f"Number of rows after dropping missing values: {len(df)}")

Number of rows before dropping missing values: 358525


/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_14393/2311168238.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]


Number of rows after dropping missing values: 355352


In [24]:
# Drop missing location_old if whole column location_new is missing and vice versa
print(f"Before dropping missing location values: {len(df)}")
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    if df_sub['location_old'].isna().all():
        df = df[~(df['survey_file'] == file) | ~df['location_new'].isna()]
    elif df_sub['location_new'].isna().all():
        df = df[~(df['survey_file'] == file) | ~df['location_old'].isna()]
print(f"After dropping missing location values: {len(df)}")

Before dropping missing location values: 355352
After dropping missing location values: 355352


In [25]:
# Save dataframe 
df.to_csv(data_root / "CBOS_data_clean.csv", index = True)